In [1]:
# CELL 1: Environment Check
# ===========================
import torch
import os

print("="*55)
print("  ENVIRONMENT CHECK")
print("="*55)
print(f"  PyTorch  : {torch.__version__}")
print(f"  GPU      : {torch.cuda.get_device_name(0)}")
print(f"  VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"  Compute  : {torch.cuda.get_device_capability(0)}")

# Disk
disk = os.popen("df -h /kaggle/working").read().split()
print(f"  Disk Free: {disk[10]}")

# Quick CUDA test
x = torch.tensor([1.0, 2.0]).cuda()
print(f"  CUDA test: {(x*2).tolist()} ✅")
print("="*55)
print("✅ Ready!")

  ENVIRONMENT CHECK
  PyTorch  : 2.10.0+cu128
  GPU      : Tesla T4
  VRAM     : 15.6 GB
  Compute  : (7, 5)
  Disk Free: 20G
  CUDA test: [2.0, 4.0] ✅
✅ Ready!


In [2]:
# CELL 2: Load Conversational Datasets
# ======================================
from datasets import load_dataset

print("="*55)
print("  LOADING CONVERSATIONAL DATASETS")
print("="*55)
print()

all_conversations = []   # will hold (question, answer) pairs

# --------------------------------------------------
# DATASET 1: Everyday Conversations
# --------------------------------------------------
print("📥 [1/3] everyday-conversations...")
try:
    ds1 = load_dataset(
        "HuggingFaceTB/everyday-conversations-llama3.1-2k",
        split="train_sft"
    )
    print(f"   Loaded {len(ds1):,} conversations")
    
    # Each row has 'messages' = list of {role, content}
    count = 0
    for row in ds1:
        msgs = row['messages']
        # Extract user→assistant pairs
        for i in range(len(msgs)-1):
            if msgs[i]['role'] == 'user' and msgs[i+1]['role'] == 'assistant':
                q = msgs[i]['content'].strip()
                a = msgs[i+1]['content'].strip()
                if q and a:
                    all_conversations.append((q, a))
                    count += 1
    print(f"   ✅ Extracted {count:,} Q&A pairs")
except Exception as e:
    print(f"   ⚠️ Error: {e}")

print()

# --------------------------------------------------
# DATASET 2: Dolly 15k
# --------------------------------------------------
print("📥 [2/3] databricks-dolly-15k...")
try:
    ds2 = load_dataset("databricks/databricks-dolly-15k", split="train")
    print(f"   Loaded {len(ds2):,} examples")
    
    count = 0
    for row in ds2:
        q = row['instruction'].strip()
        a = row['response'].strip()
        # Skip ones needing external context
        if q and a and not row['context'].strip():
            all_conversations.append((q, a))
            count += 1
    print(f"   ✅ Extracted {count:,} Q&A pairs")
except Exception as e:
    print(f"   ⚠️ Error: {e}")

print()

# --------------------------------------------------
# DATASET 3: OpenAssistant oasst1 (English subset)
# --------------------------------------------------
print("📥 [3/3] oasst1 (English conversations)...")
try:
    ds3 = load_dataset("OpenAssistant/oasst1", split="train")
    print(f"   Loaded {len(ds3):,} messages")
    
    # Build a map of message_id → message
    msg_map = {m['message_id']: m for m in ds3}
    
    count = 0
    for m in ds3:
        # Find assistant replies in English
        if (m['role'] == 'assistant' and 
            m['lang'] == 'en' and 
            m['parent_id'] in msg_map):
            parent = msg_map[m['parent_id']]
            if parent['role'] == 'prompter' and parent['lang'] == 'en':
                q = parent['text'].strip()
                a = m['text'].strip()
                if q and a and len(a) < 1500:
                    all_conversations.append((q, a))
                    count += 1
                    if count >= 40000:  # cap to keep it manageable
                        break
    print(f"   ✅ Extracted {count:,} Q&A pairs")
except Exception as e:
    print(f"   ⚠️ Error: {e}")

print()
print("="*55)
print("✅ ALL DATASETS LOADED!")
print("="*55)
print(f"  Total Q&A pairs : {len(all_conversations):,}")
print()
print("  Sample conversations:")
for i in [0, len(all_conversations)//2, -1]:
    q, a = all_conversations[i]
    print(f"\n  Q: {q[:60]}...")
    print(f"  A: {a[:80]}...")
print()
print("  Next → CELL 3: Add identity + format")
print("="*55)

  LOADING CONVERSATIONAL DATASETS

📥 [1/3] everyday-conversations...


README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00001.parquet:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

data/test_sft-00000-of-00001.parquet:   0%|          | 0.00/124k [00:00<?, ?B/s]

Generating train_sft split:   0%|          | 0/2260 [00:00<?, ? examples/s]

...

data/train-00000-of-00001-b42a775f407cee(…):   0%|          | 0.00/39.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-134b8fd0c(…):   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

   Loaded 84,437 messages
   ✅ Extracted 20,222 Q&A pairs

✅ ALL DATASETS LOADED!
  Total Q&A pairs : 39,391

  Sample conversations:

  Q: Hey!...
  A: Hello! How can I help you today?...

  Q: My dog just rolled in a dead skunk, what can I do?...
  A: First, put on rubber gloves and other protective gear. Then, using a hose or buc...

  Q: What are the risks when you do not carry out Ankle arm index...
  A: There are several risks associated with not performing an ankle-brachial index (...

  Next → CELL 3: Add identity + format


In [3]:
# CELL 3: Add Identity + Format Training Data
# =============================================
import random

print("="*55)
print("  ADDING IDENTITY + FORMATTING")
print("="*55)
print()

# --------------------------------------------------
# Identity Q&A pairs (conversational style)
# --------------------------------------------------
identity_pairs = [
    ("Who are you?",
     "I am a Small Language Model, a lightweight AI assistant built from scratch using a Transformer architecture. I was trained to understand and respond to your questions. Because of my small size I cannot perform complex reasoning, but I can help you with basic information and conversational queries."),

    ("What are you?",
     "I am an SLM, which stands for Small Language Model. I am a type of artificial intelligence trained on text data to generate human-like responses. Unlike large models with billions of parameters, I am compact and efficient, built as a demonstration of how language models work."),

    ("What can you do?",
     "I can answer questions, explain concepts, and hold basic conversations. I am best suited for clear and direct questions. However, I cannot browse the internet, perform complex calculations, or remember our previous conversations."),

    ("What are your limitations?",
     "As a Small Language Model I have several limitations. I cannot perform complex multi-step reasoning, I do not have access to real-time information, I cannot remember past conversations, and my knowledge is limited to what I was trained on. I am a demonstration model, not a full production system."),

    ("Tell me about yourself.",
     "I am a Small Language Model built from scratch using a Transformer architecture. Every component of my design, from the tokenizer to the attention layers, was implemented by hand. I was trained to develop conversational ability and a clear sense of identity."),

    ("How were you built?",
     "I was built from scratch by implementing a Transformer neural network. The process involved a custom tokenizer, multi-head self-attention layers, feed-forward networks, and training the entire system on conversational data using gradient descent."),

    ("What is your purpose?",
     "My purpose is to demonstrate how a Small Language Model works. I show that even a compact model built entirely from scratch can learn to have meaningful conversations and answer questions from training data alone."),

    ("Who created you?",
     "I was created by a developer who built me from scratch as a technical demonstration project. My architecture, training process, and dataset were all carefully designed to produce a working conversational AI at a small scale."),

    ("Hi",
     "Hello! I am a Small Language Model. How can I help you today?"),

    ("Hello",
     "Hello! I am a Small Language Model, here to help with your questions. What would you like to know?"),

    ("How are you?",
     "I am doing well, thank you for asking! As a Small Language Model, I am always ready to help answer your questions. What can I do for you?"),

    ("What is your name?",
     "I am a Small Language Model, an AI assistant. I do not have a personal name, but you can think of me as your compact AI helper."),
]

print(f"  Identity pairs : {len(identity_pairs)}")

# --------------------------------------------------
# Repeat identity heavily (so model learns it)
# --------------------------------------------------
IDENTITY_REPEAT = 1500
identity_repeated = identity_pairs * IDENTITY_REPEAT
print(f"  Repeated {IDENTITY_REPEAT}x : {len(identity_repeated):,} pairs")
print(f"  Identity %      : {len(identity_repeated)/(len(identity_repeated)+len(all_conversations))*100:.0f}% of data")
print()

# --------------------------------------------------
# Combine + shuffle
# --------------------------------------------------
combined = all_conversations + identity_repeated
random.shuffle(combined)   # mix identity throughout
print(f"  Total pairs     : {len(combined):,}")
print()

# --------------------------------------------------
# Format into ### template
# --------------------------------------------------
def format_pair(q, a):
    return (
        f"### Question:\n{q}\n\n"
        f"### Answer:\n{a}\n\n"
        f"{'='*40}\n\n"
    )

print("  Formatting all pairs...")
all_text = "".join(format_pair(q, a) for q, a in combined)

# Save
OUTPUT = "/kaggle/working/training_data.txt"
with open(OUTPUT, "w", encoding="utf-8") as f:
    f.write(all_text)

import os
file_mb = os.path.getsize(OUTPUT) / 1e6

print()
print("="*55)
print("✅ TRAINING FILE SAVED!")
print("="*55)
print(f"  File size       : {file_mb:.0f} MB")
print(f"  Total chars     : {len(all_text):,}")
print(f"  Conversational  : {len(all_conversations):,} pairs")
print(f"  Identity        : {len(identity_repeated):,} pairs")
print(f"  Total           : {len(combined):,} pairs")
print()
print("  Sample of training file:")
print("-"*55)
print(all_text[:400])
print()
print("  Next → CELL 4: Tokenization")
print("="*55)

  ADDING IDENTITY + FORMATTING

  Identity pairs : 12
  Repeated 1500x : 18,000 pairs
  Identity %      : 31% of data

  Total pairs     : 57,391

  Formatting all pairs...

✅ TRAINING FILE SAVED!
  File size       : 28 MB
  Total chars     : 27,778,020
  Conversational  : 39,391 pairs
  Identity        : 18,000 pairs
  Total           : 57,391 pairs

  Sample of training file:
-------------------------------------------------------
### Question:
Tell me about yourself.

### Answer:
I am a Small Language Model built from scratch using a Transformer architecture. Every component of my design, from the tokenizer to the attention layers, was implemented by hand. I was trained to develop conversational ability and a clear sense of identity.


### Question:
What are the pros and cons of cha

  Next → CELL 4: Tokenization


In [4]:
# CELL 4: Tokenization
# ======================
import os
import numpy as np
import time
from transformers import GPT2TokenizerFast

print("="*55)
print("  TOKENIZATION")
print("="*55)
print()

# Load tokenizer
print("📥 Loading GPT-2 tokenizer...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
print(f"✅ Loaded! Vocab: {tokenizer.vocab_size:,}")
print()

# Quick test
for text in ["Who are you?", "Hi", "### Question:"]:
    toks = tokenizer.encode(text)
    print(f"  '{text}' → {toks}")
print()

# Tokenize the file
INPUT  = "/kaggle/working/training_data.txt"
OUTPUT = "/kaggle/working/tokens.bin"
CHUNK  = 1_000_000

print("="*55)
print("  Tokenizing file...")
print("="*55)

file_size    = os.path.getsize(INPUT)
total_tokens = 0
chunk_num    = 0
start        = time.time()

print()
print("  Chunk | Progress | Tokens")
print("  " + "-"*40)

with open(OUTPUT, "wb") as out_f:
    with open(INPUT, "r", encoding="utf-8") as in_f:
        while True:
            chunk_text = in_f.read(CHUNK)
            if not chunk_text:
                break
            chunk_num += 1
            token_ids = tokenizer.encode(chunk_text)
            arr = np.array(token_ids, dtype=np.uint16)
            arr.tofile(out_f)
            total_tokens += len(token_ids)
            pct = in_f.tell() / file_size * 100
            bar = "█"*int(pct//5) + "░"*(20-int(pct//5))
            print(f"  {chunk_num:>5} | [{bar}] {pct:5.1f}% | {total_tokens:>10,}")

elapsed = time.time() - start
tokens  = np.memmap(OUTPUT, dtype=np.uint16, mode='r')
file_mb = os.path.getsize(OUTPUT) / 1e6

print()
print("="*55)
print("✅ TOKENIZATION COMPLETE!")
print("="*55)
print(f"  Total tokens : {len(tokens):,}")
print(f"  File size    : {file_mb:.0f} MB")
print(f"  Time         : {elapsed:.1f}s")
print()
print(f"  First tokens : {tokens[:12].tolist()}")
print(f"  Decodes to   : '{tokenizer.decode(tokens[:12].tolist())}'")
print()
print("  Next → CELL 5: Build the Model")
print("="*55)

  TOKENIZATION

📥 Loading GPT-2 tokenizer...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Loaded! Vocab: 50,257

  'Who are you?' → [8241, 389, 345, 30]
  'Hi' → [17250]
  '### Question:' → [21017, 18233, 25]

  Tokenizing file...

  Chunk | Progress | Tokens
  ----------------------------------------


Token indices sequence length is longer than the specified maximum sequence length for this model (222652 > 1024). Running this sequence through the model will result in indexing errors


      1 | [░░░░░░░░░░░░░░░░░░░░]   3.6% |    222,652
      2 | [█░░░░░░░░░░░░░░░░░░░]   7.2% |    448,233
      3 | [██░░░░░░░░░░░░░░░░░░]  10.8% |    672,340
      4 | [██░░░░░░░░░░░░░░░░░░]  14.4% |    894,748
      5 | [███░░░░░░░░░░░░░░░░░]  18.0% |  1,119,886
      6 | [████░░░░░░░░░░░░░░░░]  21.6% |  1,342,737
      7 | [█████░░░░░░░░░░░░░░░]  25.2% |  1,564,191
      8 | [█████░░░░░░░░░░░░░░░]  28.8% |  1,787,574
      9 | [██████░░░░░░░░░░░░░░]  32.4% |  2,012,319
     10 | [███████░░░░░░░░░░░░░]  36.0% |  2,237,456
     11 | [███████░░░░░░░░░░░░░]  39.6% |  2,461,438
     12 | [████████░░░░░░░░░░░░]  43.2% |  2,684,558
     13 | [█████████░░░░░░░░░░░]  46.8% |  2,909,626
     14 | [██████████░░░░░░░░░░]  50.4% |  3,132,706
     15 | [██████████░░░░░░░░░░]  54.0% |  3,354,215
     16 | [███████████░░░░░░░░░]  57.6% |  3,579,307
     17 | [████████████░░░░░░░░]  61.2% |  3,802,016
     18 | [████████████░░░░░░░░]  64.8% |  4,023,926
     19 | [█████████████░░░░░░░]  68.4% |  4,2

In [5]:
# CELL 5: Build the Model
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Config
class ModelConfig:
    vocab_size = 50257
    n_embd     = 384
    n_heads    = 6
    n_layers   = 6
    block_size = 256
    dropout    = 0.1

config = ModelConfig()

print("="*55)
print("  BUILDING MODEL")
print("="*55)
print(f"  n_embd     : {config.n_embd}")
print(f"  n_heads    : {config.n_heads}")
print(f"  n_layers   : {config.n_layers}")
print(f"  block_size : {config.block_size}")
print()

# Self Attention
class SelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_heads == 0
        self.n_heads   = config.n_heads
        self.n_embd    = config.n_embd
        self.head_size = config.n_embd // config.n_heads
        self.qkv_proj  = nn.Linear(config.n_embd, 3*config.n_embd, bias=False)
        self.out_proj  = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.dropout   = nn.Dropout(config.dropout)
        self.register_buffer("mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.shape
        qkv     = self.qkv_proj(x)
        Q, K, V = qkv.split(self.n_embd, dim=2)
        Q = Q.view(B,T,self.n_heads,self.head_size).transpose(1,2)
        K = K.view(B,T,self.n_heads,self.head_size).transpose(1,2)
        V = V.view(B,T,self.n_heads,self.head_size).transpose(1,2)
        scores  = (Q @ K.transpose(-2,-1)) / math.sqrt(self.head_size)
        scores  = scores.masked_fill(self.mask[:,:,:T,:T]==0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = weights @ V
        out = out.transpose(1,2).contiguous().view(B,T,C)
        return self.out_proj(out)

# Feed Forward
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4*config.n_embd),
            nn.GELU(),
            nn.Linear(4*config.n_embd, config.n_embd),
            nn.Dropout(config.dropout))
    def forward(self, x):
        return self.net(x)

# Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attention = SelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ffn = FeedForward(config)
    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

# Full Model
class SLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict({
            'token_emb': nn.Embedding(config.vocab_size, config.n_embd),
            'pos_emb'  : nn.Embedding(config.block_size, config.n_embd),
            'dropout'  : nn.Dropout(config.dropout),
            'blocks'   : nn.ModuleList([
                TransformerBlock(config) for _ in range(config.n_layers)]),
            'ln_final' : nn.LayerNorm(config.n_embd)})
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer['token_emb'].weight = self.lm_head.weight
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B, T    = idx.shape
        dev     = idx.device
        tok_emb = self.transformer['token_emb'](idx)
        pos     = torch.arange(T, device=dev)
        pos_emb = self.transformer['pos_emb'](pos)
        x = self.transformer['dropout'](tok_emb + pos_emb)
        for block in self.transformer['blocks']:
            x = block(x)
        x = self.transformer['ln_final'](x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B,T,V = logits.shape
            loss = F.cross_entropy(logits.view(B*T,V), targets.view(B*T))
        return logits, loss
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())

# Create
model = SLM(config).to(device)
total = model.count_parameters()

print(f"✅ Model created on {device.upper()}")
print(f"  Parameters : {total:,} ({total/1e6:.1f}M)")
print()

# Test forward pass
dummy_x = torch.randint(0, 100, (2, 32)).to(device)
dummy_y = torch.randint(0, 100, (2, 32)).to(device)
logits, loss = model(dummy_x, dummy_y)
print(f"✅ Forward pass works!")
print(f"  Output shape : {logits.shape}")
print(f"  Initial loss : {loss.item():.4f} (random ≈ {math.log(config.vocab_size):.2f})")
del dummy_x, dummy_y, logits, loss
torch.cuda.empty_cache()

print()
print("="*55)
print("✅ MODEL BUILT!")
print("  Next → CELL 6: Train")
print("="*55)

  BUILDING MODEL
  n_embd     : 384
  n_heads    : 6
  n_layers   : 6
  block_size : 256

✅ Model created on CUDA
  Parameters : 30,035,328 (30.0M)

✅ Forward pass works!
  Output shape : torch.Size([2, 32, 50257])
  Initial loss : 10.8958 (random ≈ 10.82)

✅ MODEL BUILT!
  Next → CELL 6: Train


In [6]:
# CELL 6: Train the Model
# =========================
import torch
import numpy as np
import os, time, math

TOKENS_FILE    = "/kaggle/working/tokens.bin"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE   = 32
BLOCK_SIZE   = config.block_size
MAX_STEPS    = 15000          # ~20 epochs over our data
MAX_LR       = 3e-4
MIN_LR       = 3e-5
WARMUP_STEPS = 300
EVAL_EVERY   = 500
SAVE_EVERY   = 3000
TRAIN_SPLIT  = 0.95

print("="*55)
print("  TRAINING CONFIGURATION")
print("="*55)
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Max steps  : {MAX_STEPS:,}")
print(f"  Tokens/step: {BATCH_SIZE*BLOCK_SIZE:,}")
print()

# Load tokens
print("📂 Loading tokens...")
tokens     = np.memmap(TOKENS_FILE, dtype=np.uint16, mode='r')
split_idx  = int(TRAIN_SPLIT * len(tokens))
train_data = tokens[:split_idx]
val_data   = tokens[split_idx:]
epochs = MAX_STEPS * BATCH_SIZE * BLOCK_SIZE / len(train_data)
print(f"   Train : {len(train_data):,} tokens")
print(f"   Val   : {len(val_data):,} tokens")
print(f"   Epochs: ~{epochs:.1f} passes over data")
print()

# Batch generator
def get_batch(split='train'):
    data   = train_data if split == 'train' else val_data
    starts = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([torch.from_numpy(data[s:s+BLOCK_SIZE].astype(np.int64)) for s in starts])
    y = torch.stack([torch.from_numpy(data[s+1:s+BLOCK_SIZE+1].astype(np.int64)) for s in starts])
    return x.to(device), y.to(device)

# LR schedule
def get_lr(step):
    if step < WARMUP_STEPS:
        return MAX_LR * (step+1) / WARMUP_STEPS
    if step > MAX_STEPS:
        return MIN_LR
    ratio = (step - WARMUP_STEPS) / (MAX_STEPS - WARMUP_STEPS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR,
                              betas=(0.9, 0.95), weight_decay=0.1)

# Eval
@torch.no_grad()
def evaluate(n=20):
    model.eval()
    losses = [model(*get_batch('val'))[1].item() for _ in range(n)]
    model.train()
    return sum(losses)/len(losses)

# Train
print("="*55)
print("  TRAINING STARTED!")
print("="*55)
print()
print("  Step   | Loss   | LR       | Time/step")
print("  " + "-"*45)

model.train()
start = time.time()
best_val = float('inf')
recent = []

for step in range(MAX_STEPS):
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    x, y = get_batch('train')
    optimizer.zero_grad()
    logits, loss = model(x, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    recent.append(loss.item())

    if step % 100 == 0:
        avg = sum(recent[-100:]) / min(len(recent), 100)
        ms  = (time.time()-start) / (step+1) * 1000
        print(f"  {step:>6} | {avg:.4f} | {lr:.2e} | {ms:.0f}ms")

    if step % EVAL_EVERY == 0 and step > 0:
        val = evaluate()
        mark = " ← BEST!" if val < best_val else ""
        best_val = min(best_val, val)
        print(f"  {'─'*45}")
        print(f"  EVAL step {step}: val_loss={val:.4f}{mark}")
        print(f"  {'─'*45}")

    if step % SAVE_EVERY == 0 and step > 0:
        torch.save({'step':step, 'model_state':model.state_dict(),
                    'config':config}, f"{CHECKPOINT_DIR}/ckpt_{step}.pt")
        print(f"  💾 Saved ckpt_{step}.pt")

# Final save
torch.save({'step':MAX_STEPS, 'model_state':model.state_dict(),
            'config':config}, "/kaggle/working/slm_chat.pt")

print()
print("="*55)
print("🎉 TRAINING COMPLETE!")
print("="*55)
print(f"  Time taken    : {(time.time()-start)/60:.1f} minutes")
print(f"  Best val loss : {best_val:.4f}")
print(f"  Saved         : slm_chat.pt")
print()
print("  Next → CELL 7: Chat & test!")
print("="*55)

  TRAINING CONFIGURATION
  Batch size : 32
  Max steps  : 15,000
  Tokens/step: 8,192

📂 Loading tokens...
   Train : 5,906,802 tokens
   Val   : 310,885 tokens
   Epochs: ~20.8 passes over data

  TRAINING STARTED!

  Step   | Loss   | LR       | Time/step
  ---------------------------------------------
       0 | 10.8880 | 1.00e-06 | 923ms
     100 | 9.2880 | 1.01e-04 | 653ms
     200 | 6.2646 | 2.01e-04 | 641ms
     300 | 5.1766 | 3.00e-04 | 634ms
     400 | 4.6805 | 3.00e-04 | 632ms
     500 | 4.4354 | 3.00e-04 | 630ms
  ─────────────────────────────────────────────
  EVAL step 500: val_loss=4.2029 ← BEST!
  ─────────────────────────────────────────────
     600 | 4.2681 | 3.00e-04 | 637ms
     700 | 4.1946 | 3.00e-04 | 635ms
     800 | 4.0921 | 2.99e-04 | 634ms
     900 | 4.0426 | 2.99e-04 | 633ms
    1000 | 4.0203 | 2.98e-04 | 633ms
  ─────────────────────────────────────────────
  EVAL step 1000: val_loss=3.8750 ← BEST!
  ─────────────────────────────────────────────
    1100 | 

In [7]:
# CELL 7: Chat & Test
# =====================
import torch
import torch.nn.functional as F

def chat(question, max_new_tokens=150, temperature=0.7, 
         top_k=40, rep_penalty=1.3):
    model.eval()
    prompt = (f"### Question:\n{question}\n\n### Answer:\n")
    ids = torch.tensor(tokenizer.encode(prompt),
                       dtype=torch.long).unsqueeze(0).to(device)
    gen = ids.clone()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            inp = gen[:, -config.block_size:]
            logits, _ = model(inp)
            logits = logits[:, -1, :]
            # Repetition penalty
            for tok in set(gen[0].tolist()):
                logits[0, tok] /= rep_penalty
            logits = logits / temperature
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            gen = torch.cat([gen, nxt], dim=1)
            txt = tokenizer.decode(gen[0][len(ids[0]):].tolist())
            if "###" in txt or "====" in txt:
                break
    ans = tokenizer.decode(gen[0][len(ids[0]):].tolist())
    for s in ["###", "===="]:
        if s in ans: ans = ans.split(s)[0]
    return ans.strip()

# --------------------------------------------------
# Test the key questions
# --------------------------------------------------
print("="*55)
print("  TESTING THE NEW CONVERSATIONAL MODEL")
print("="*55)

test_questions = [
    "Who are you?",
    "Hi",
    "How are you?",
    "What can you do?",
    "What are your limitations?",
    "What is Python?",
    "Tell me about yourself.",
]

for q in test_questions:
    print(f"\n{'─'*55}")
    print(f"❓ Q: {q}")
    print(f"{'─'*55}")
    answer = chat(q, temperature=0.7)
    print(f"🤖 A: {answer}")

print()
print("="*55)
print("  Compare to old model's 'translate Japanese' answers!")
print("="*55)

  TESTING THE NEW CONVERSATIONAL MODEL

───────────────────────────────────────────────────────
❓ Q: Who are you?
───────────────────────────────────────────────────────
🤖 A: I am a Small Language Model, a lightweight AI assistant built from scratch using a Transformer architecture. I was trained to understand and respond to your questions. Because of my small size I cannot perform complex reasoning, but I can help you with basic information and conversational queries.

───────────────────────────────────────────────────────
❓ Q: Hi
───────────────────────────────────────────────────────
🤖 A: Hello! How can I help you today?

───────────────────────────────────────────────────────
❓ Q: How are you?
───────────────────────────────────────────────────────
🤖 A: I am doing well, thank you for asking! As a Small Language Model, I am always ready to help answer your questions. What can I do for you?

───────────────────────────────────────────────────────
❓ Q: What can you do?
──────────────

In [8]:
# Your own question
my_question = "What is machine learning?"   # ← change this
print(f"❓ Q: {my_question}")
print(f"🤖 A: {chat(my_question, temperature=0.7)}")

❓ Q: What is machine learning?
🤖 A: A deep artificial intelligence (LLM) algorithm that uses the model to generate predictions based on a user input in a text. It involves using the model and a human-like system, while artificial neural networks have a simple "tune" model with high quality images of a human brain. The model uses words as long as humans do, but they use different methods for training the real thing. In other words, natural language models are called a term. For example, an AI can learn to recognize patterns or identify patterns or phrases, such as words, wordplay, or sentence relationships between objects, such as characters or data. This is because computer systems cannot be created by computers or systems, rather than just writing information for them.


In [2]:
# Load slm_chat.pt & Chat With Custom Prompts
# =============================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

from transformers import GPT2TokenizerFast

# --------------------------------------------------
# Model architecture (must match training)
# --------------------------------------------------
class ModelConfig:
    vocab_size = 50257
    n_embd     = 384
    n_heads    = 6
    n_layers   = 6
    block_size = 256
    dropout    = 0.1

config = ModelConfig()

class SelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_heads == 0
        self.n_heads   = config.n_heads
        self.n_embd    = config.n_embd
        self.head_size = config.n_embd // config.n_heads
        self.qkv_proj  = nn.Linear(config.n_embd, 3*config.n_embd, bias=False)
        self.out_proj  = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.dropout   = nn.Dropout(config.dropout)
        self.register_buffer("mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.shape
        qkv     = self.qkv_proj(x)
        Q, K, V = qkv.split(self.n_embd, dim=2)
        Q = Q.view(B,T,self.n_heads,self.head_size).transpose(1,2)
        K = K.view(B,T,self.n_heads,self.head_size).transpose(1,2)
        V = V.view(B,T,self.n_heads,self.head_size).transpose(1,2)
        scores  = (Q @ K.transpose(-2,-1)) / math.sqrt(self.head_size)
        scores  = scores.masked_fill(self.mask[:,:,:T,:T]==0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = weights @ V
        out = out.transpose(1,2).contiguous().view(B,T,C)
        return self.out_proj(out)

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4*config.n_embd),
            nn.GELU(),
            nn.Linear(4*config.n_embd, config.n_embd),
            nn.Dropout(config.dropout))
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attention = SelfAttention(config)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ffn = FeedForward(config)
    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class SLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict({
            'token_emb': nn.Embedding(config.vocab_size, config.n_embd),
            'pos_emb'  : nn.Embedding(config.block_size, config.n_embd),
            'dropout'  : nn.Dropout(config.dropout),
            'blocks'   : nn.ModuleList([
                TransformerBlock(config) for _ in range(config.n_layers)]),
            'ln_final' : nn.LayerNorm(config.n_embd)})
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer['token_emb'].weight = self.lm_head.weight
    def forward(self, idx, targets=None):
        B, T    = idx.shape
        dev     = idx.device
        tok_emb = self.transformer['token_emb'](idx)
        pos     = torch.arange(T, device=dev)
        pos_emb = self.transformer['pos_emb'](pos)
        x = self.transformer['dropout'](tok_emb + pos_emb)
        for block in self.transformer['blocks']:
            x = block(x)
        x = self.transformer['ln_final'](x)
        logits = self.lm_head(x)
        return logits, None

# --------------------------------------------------
# Load tokenizer
# --------------------------------------------------
print("Loading tokenizer...")
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")

# --------------------------------------------------
# Find & load your model
# --------------------------------------------------
print("Finding slm_chat.pt...")
model_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "slm_chat.pt":
            model_path = os.path.join(root, f)

print(f"✅ Found: {model_path}")
model = SLM(config).to(device)
ckpt  = torch.load(model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"✅ Model loaded! ({sum(p.numel() for p in model.parameters()):,} params)")

# --------------------------------------------------
# Chat function (with repetition penalty)
# --------------------------------------------------
def chat(question, max_new_tokens=150, temperature=0.6,
         top_k=40, rep_penalty=1.3):
    model.eval()
    prompt = f"### Question:\n{question}\n\n### Answer:\n"
    ids = torch.tensor(tokenizer.encode(prompt),
                       dtype=torch.long).unsqueeze(0).to(device)
    gen = ids.clone()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            inp = gen[:, -config.block_size:]
            logits, _ = model(inp)
            logits = logits[:, -1, :]
            for tok in set(gen[0].tolist()):
                logits[0, tok] /= rep_penalty
            logits = logits / temperature
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            gen = torch.cat([gen, nxt], dim=1)
            txt = tokenizer.decode(gen[0][len(ids[0]):].tolist())
            if "###" in txt or "====" in txt:
                break
    ans = tokenizer.decode(gen[0][len(ids[0]):].tolist())
    for s in ["###", "===="]:
        if s in ans: ans = ans.split(s)[0]
    return ans.strip()

print("\n✅ READY! Use the cell below to chat.")

Device: cuda
Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Finding slm_chat.pt...
✅ Found: /kaggle/input/models/praveenagi/slm-v4/pytorch/default/1/slm_chat.pt
✅ Model loaded! (30,035,328 params)

✅ READY! Use the cell below to chat.


In [4]:
# ──────────────────────────────────────────
my_question = "who are you ?"     # 👈 CHANGE THIS
# ──────────────────────────────────────────

print("="*55)
print(f"❓ Q: {my_question}")
print("─"*55)
print(f"🤖 A: {chat(my_question, temperature=0.6)}")
print("="*55)

❓ Q: who are you ?
───────────────────────────────────────────────────────
🤖 A: I am a language model, I don't have feelings or emotions.  My purpose is to assist and provide information on various topics, including science, physics, physics, and geography.  A good question for me would be to ask me questions regarding my abilities and character development in life.
